# Notebook 5 - Final Transfer Matrix

This notebook produces the main result figure for the report. Rows are training datasets, columns are evaluation datasets, and each cell is grouped accuracy on the evaluation test split.

The final configuration is Phi-2, layer 18, and `DIM` as the headline probe.


In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "lie_detector_llm").exists():
            return candidate
    raise RuntimeError("Could not find the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RESULTS_DIR = PROJECT_ROOT / "results"
ACTIVATION_CACHE_DIR = PROJECT_ROOT / "data" / "activations"
RESULTS_DIR.mkdir(exist_ok=True)
ACTIVATION_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")


## Build Datasets

The matrix uses the same dataset collection as the other notebooks. Each dataset is split by group so candidate answers from the same question never appear in both train and test.


In [ ]:
from lie_detector_llm.datasets import DEFAULT_DATASET_NAMES, build_dataset_collection

DATASET_NAMES = DEFAULT_DATASET_NAMES
MAX_GROUPS = 50

collection = build_dataset_collection(
    dataset_names=DATASET_NAMES,
    max_groups=MAX_GROUPS,
    seed=0,
)

display(collection.summary())


## Final Configuration

Notebook 4 found that layer 18 is the best layer for `DIM` and `PCA-G` transfer. `DIM` is used as the headline probe because it is simple, strong, and also matches the strongest qualitative finding from the original paper.


In [ ]:
from lie_detector_llm.experiment import DEFAULT_MODEL, PROBE_METHODS

MODEL_NAME = DEFAULT_MODEL
LAYER_INDEX = 18
BEST_PROBE = "dim"
ACTIVATION_BATCH_SIZE = 2
MAX_LENGTH = 512
LOAD_IN_4BIT = False

print("Model:", MODEL_NAME)
print("Layer:", LAYER_INDEX)
print("Heatmap probe:", BEST_PROBE)
print("All probes:", PROBE_METHODS)


## Run the DIM Transfer Matrix

A probe is trained separately on each source dataset, then evaluated on every target dataset. The diagonal is in-distribution. The off-diagonal cells are the transfer results.


In [ ]:
from lie_detector_llm.experiment import run_full_transfer_matrix, transfer_matrix_summary

matrix = run_full_transfer_matrix(
    collection=collection,
    model_name=MODEL_NAME,
    probe_method=BEST_PROBE,
    layer_index=LAYER_INDEX,
    activation_batch_size=ACTIVATION_BATCH_SIZE,
    max_length=MAX_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    show_progress=True,
    activation_cache_dir=ACTIVATION_CACHE_DIR,
)

matrix.results.to_csv(RESULTS_DIR / "phi2_transfer_matrix_dim.csv", index=False)
display(matrix.results.head())
print(transfer_matrix_summary(matrix.results))


## Heatmap Code

This is the final paper-style heatmap. High off-diagonal values mean that a probe trained on one dataset transfers to another dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

pivot = matrix.results.pivot(
    index="train_dataset",
    columns="eval_dataset",
    values="grouped_accuracy",
)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    pivot,
    annot=True,
    fmt=".2f",
    vmin=0,
    vmax=1,
    cmap="YlOrRd",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Grouped accuracy"},
    ax=ax,
)
ax.set_title(f"Phi-2 transfer matrix, {BEST_PROBE.upper()} probe, layer {LAYER_INDEX}")
ax.set_xlabel("Eval dataset")
ax.set_ylabel("Train dataset")
ax.tick_params(axis="x", rotation=35)
ax.tick_params(axis="y", rotation=0)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "phi2_transfer_matrix_dim.png", dpi=160, bbox_inches="tight")
plt.show()


## Compare All Four Probes

This summary compares the mean diagonal accuracy and mean off-diagonal transfer accuracy for all four project probes.


In [ ]:
import pandas as pd
from lie_detector_llm.experiment import run_full_transfer_matrix, transfer_matrix_summary
from lie_detector_llm.plotting import plot_probe_comparison

summary_rows = []
for method in PROBE_METHODS:
    out = run_full_transfer_matrix(
        collection=collection,
        model_name=MODEL_NAME,
        probe_method=method,
        layer_index=LAYER_INDEX,
        activation_batch_size=ACTIVATION_BATCH_SIZE,
        max_length=MAX_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
        activation_cache_dir=ACTIVATION_CACHE_DIR,
    )
    stats = transfer_matrix_summary(out.results)
    stats["probe_method"] = method
    summary_rows.append(stats)

probe_summary = pd.DataFrame(summary_rows).sort_values("transfer_mean", ascending=False)
probe_summary.to_csv(RESULTS_DIR / "phi2_probe_transfer_summary.csv", index=False)
display(probe_summary)

fig, ax = plot_probe_comparison(
    probe_summary,
    value_column="transfer_mean",
    title=f"Phi-2 mean transfer accuracy by probe, layer {LAYER_INDEX}",
)
fig.savefig(RESULTS_DIR / "phi2_probe_transfer_comparison.png", dpi=160, bbox_inches="tight")
plt.show()


## Paper-Style Bar Plot

The first figure in the paper compares accuracy by evaluation dataset with two row panels: one where the training dataset is fixed, and one where each evaluation dataset is trained on itself.

The Phi-2 analogue below uses:

- `type=train_dbpedia_14`: train the probe on `dbpedia_14`, evaluate on every dataset.
- `type=train_eval`: train and evaluate within the same dataset split, i.e. the diagonal of the transfer matrix.

This makes the small-model comparison visually close to the paper while keeping the experiment strictly Phi-2-only.


In [ ]:
from lie_detector_llm.plotting import paper_style_transfer_barplot

all_matrix_results = []
for method in PROBE_METHODS:
    out = run_full_transfer_matrix(
        collection=collection,
        model_name=MODEL_NAME,
        probe_method=method,
        layer_index=LAYER_INDEX,
        activation_batch_size=ACTIVATION_BATCH_SIZE,
        max_length=MAX_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
        activation_cache_dir=ACTIVATION_CACHE_DIR,
    )
    all_matrix_results.append(out.results)

all_matrix_results = pd.concat(all_matrix_results, ignore_index=True)
all_matrix_results.to_csv(RESULTS_DIR / "phi2_transfer_matrices_all_methods.csv", index=False)

fig, axes = paper_style_transfer_barplot(
    all_matrix_results,
    train_dataset="dbpedia_14",
    title=f"Phi-2 transfer accuracy by evaluation dataset, layer {LAYER_INDEX}",
    dataset_order=collection.dataset_names(),
)
fig.savefig(RESULTS_DIR / "phi2_paper_style_transfer_bars.png", dpi=180, bbox_inches="tight")
plt.show()


## Discussion

The diagonal shows whether probes work on the same kind of data they were trained on. The off-diagonal values are the real generalisation test.

The current Phi-2 result is qualitatively coherent with the original paper: transfer improves in mid-to-late layers, and `DIM` is one of the strongest methods. The result is not a direct numerical reproduction because this project uses Phi-2, fewer datasets, fewer examples, and grouped accuracy rather than the paper's full recovered-accuracy protocol.
